<a href="https://colab.research.google.com/github/alperarslan19/asthma-voice-dl/blob/main/notebooks/20_smoke_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 20 — Faz 2 smoke test'leri (SMK-001)

**Bilimsel amaç** (D-034 madde 10; `docs/PHASE2_DESIGN.md` Bölüm 5.1): gömme çıkarımına başlamadan önce her backbone için şunları göstermek:
- resmi ağırlık eksiksiz yükleniyor (`strict=True`, sha256);
- model resmi girdi biçimini alıyor (SR, frontend, `do_normalize`; `configs/model_input_contracts.yaml`);
- eval modunda deterministik ve batch'ten bağımsız (katılımcılar arası bilgi akışı yok);
- çıktı anlamlı: PANNs kelime kayıtlarında AudioSet "Speech" sınıfını üste koyuyor; bütün modellerde aynı kişinin kayıtları birbirine daha benzer;
- Faz 3 için: eğitim modunda batch 16 × 4 s, fp16, bir adımın GPU belleği ve süresi.

**Çalışma ortamı:** *Runtime → Change runtime type → T4 GPU*. Süre: ağırlık indirme ~10 dk (ilk sefer), test ~10 dk, smoke test ~10 dk. *Run all* kullanılabilir: bir adım başarısız olursa notebook orada durur.

**Önce yapılacak (bir kez, elle):** BEATs ağırlığı OneDrive'da; otomatik indirilemiyor. `github.com/microsoft/unilm/tree/master/beats` README tablosunda **"BEATs_iter3+ (AS2M)"** (fine-tuned *olmayan*, üçüncü sütun) bağlantısından indir ve Drive'da `MyDrive/asthma-voice/models/BEATs_iter3_plus_AS2M.pt` adıyla kaydet.

Herhangi bir backbone FAIL verirse **21 numaralı notebook'a geçme**. Rapor yine GitHub'a gönderilir (olumsuz sonuç korunur), son hücre FAIL'leri listeler ve durur. Önce nedeni incele (rapor JSON'undaki değerler), düzelt, EXPERIMENTS.md'deki SMK-001 kaydına yaz ve SMK-001'i baştan çalıştır; eşik gevşeterek geçirme (`docs/PHASE2_DESIGN.md` 1.1). Raporu birlikte inceleyelim.

## 0. Oturum kurulumu

In [ ]:
import shutil, subprocess, platform, sys
print('Python:', sys.version.split()[0], '|', platform.platform())
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                         capture_output=True, text=True).stdout)
else:
    print('UYARI: GPU yok (CPU oturumu). SMK-001 (Bölüm 3) GPU ister: Runtime → Change runtime type → T4 GPU.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess, sys, os, json, shutil, hashlib, codecs, platform
DRIVE_ROOT  = Path('/content/drive/MyDrive/asthma-voice')
DERIVED     = DRIVE_ROOT / 'data_derived'
MODELS      = DRIVE_ROOT / 'models'
EXPERIMENTS = DRIVE_ROOT / 'experiments'
DRIVE_CACHE = DERIVED / 'audio_cache_v1'
LOCAL_CACHE = Path('/content/audio_cache_v1')
EMB_DIR     = DERIVED / 'embeddings_v1'
GITHUB_USER, REPO_NAME = 'alperarslan19', 'asthma-voice-dl'
REPO_DIR    = Path('/content') / REPO_NAME
BACKBONES   = ['cnn10', 'cnn14', 'cnn14_16k', 'beats', 'wavlm_base_plus', 'wavlm_large']
EXPECTED_N_RECORDINGS, EXPECTED_N_PARTICIPANTS = 2393, 342   # Faz 1 önbellek kohortu (reports/audio_cache)
for p in [DERIVED / 'participant_context.csv', DERIVED / 'splits' / 'outer_r0.csv', DRIVE_CACHE / 'cache_info.json']:
    assert p.exists(), f'Bulunamadı: {p}'
for p in [MODELS, EXPERIMENTS, EMB_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def sh(cmd, check=True):
    """Kabuk komutunu çalıştırır, çıktıyı canlı yazar. check=True iken komut başarısızsa hücreyi DURDURUR.
    (`!komut` başarısız olsa da hücre devam eder; 'Run all' hatayı fark etmeden sonraki adıma geçerdi.)"""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    dec = codecs.getincrementaldecoder('utf-8')('replace')
    while True:
        b = p.stdout.read1(1 << 14)
        if not b:
            break
        sys.stdout.write(dec.decode(b))
        sys.stdout.flush()
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f'Komut başarısız (çıkış kodu {rc}): {cmd[:300]}')
    return None if check else rc


print('Girdiler mevcut.')

In [ ]:
# Repo: yoksa klonla; varsa GitHub'daki main'e güvenli biçimde geç (önceki çalıştırmanın rapor dalında kalmış olabilir)
from google.colab import userdata
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception as e:
    raise RuntimeError("GITHUB_TOKEN okunamadı: sol paneldeki anahtar (Secrets) simgesinden GITHUB_TOKEN ekli mi ve "
                       "'Notebook access' açık mı?") from e
assert token, 'GITHUB_TOKEN boş'
REMOTE = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'


def git(*args, check=True, show=False):
    """git komutu; çıktıda token gizlenir. check=True iken hata durumunda hücreyi durdurur."""
    r = subprocess.run(['git', '-C', str(REPO_DIR), *args], capture_output=True, text=True)
    r.stdout, r.stderr = r.stdout.replace(token, '***'), r.stderr.replace(token, '***')
    if show and (r.stdout + r.stderr).strip():
        print((r.stdout + r.stderr).strip())
    if check and r.returncode != 0:
        cmd = ' '.join(args[:3]).replace(token, '***')
        raise RuntimeError(f'git {cmd} başarısız (çıkış {r.returncode}):\n{(r.stdout + r.stderr).strip()}')
    return r


if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    raise RuntimeError(f'{REPO_DIR} var ama git deposu değil (yarım kalmış klon?). Şunu çalıştırıp bu hücreyi yeniden '
                       f'çalıştır:  !rm -rf {REPO_DIR}')
if not REPO_DIR.exists():
    r = subprocess.run(['git', 'clone', REMOTE, str(REPO_DIR)], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone başarısız:\n' + (r.stdout + r.stderr).replace(token, '***'))
    print('Klonlandı.')
else:
    git('remote', 'set-url', 'origin', REMOTE)            # token yenilendiyse
    git('fetch', '--prune', 'origin')
    branch = git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip()
    unpushed = git('log', 'HEAD', '--not', '--remotes', '--oneline').stdout.strip()
    dirty = git('status', '--porcelain', '--untracked-files=all').stdout.strip()
    if unpushed:
        raise RuntimeError(
            f"'{branch}' dalında GitHub'a gönderilmemiş commit var:\n{unpushed}\n"
            "Bu, yarım kalmış bir rapor gönderimidir; raporu kaybetmemek için önce 'Raporu GitHub'a gönder' hücresini "
            "çalıştır (aynı commit'i yeniden gönderir), sonra bu hücreyi yeniden çalıştır.")
    if dirty:
        at_main = branch == 'main' and git('rev-parse', 'HEAD').stdout == git('rev-parse', 'origin/main').stdout
        if not at_main:
            raise RuntimeError(
                f"Repo klasöründe commit edilmemiş dosyalar var (dal: {branch}):\n{dirty}\n"
                "Bunlar önceki bir çalıştırmanın raporlarıysa önce gönder hücresini çalıştır. Gerekmiyorsa klasörü silip "
                f"bu hücreyi yeniden çalıştır:  !rm -rf {REPO_DIR}")
        print('UYARI: main zaten GitHub ile aynı; önceki çalıştırmadan kalan commit edilmemiş dosyalara dokunulmadı:\n' + dirty)
    else:
        git('checkout', '-B', 'main', 'origin/main')       # temiz klasör: main = GitHub'daki main (yerel diğer dallar kalır)
        if branch != 'main':
            print(f"'{branch}' dalından main'e geçildi (o dal GitHub'da duruyor).")
assert git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip() == 'main'
COMMIT = git('rev-parse', 'HEAD').stdout.strip()
assert COMMIT == git('rev-parse', 'origin/main').stdout.strip(), 'yerel main GitHub main ile aynı değil'
print('Dal: main | commit:', COMMIT[:12], '(= origin/main)')

In [ ]:
sh(f'pip install -q -r {REPO_DIR}/requirements-colab.txt')
import torch, torchaudio, transformers, torchlibrosa, sklearn, numpy as np
print('torch', torch.__version__, '| torchaudio', torchaudio.__version__, '| transformers', transformers.__version__,
      '| sklearn', sklearn.__version__, '| numpy', np.__version__)
import torchaudio.compliance.kaldi  # BEATs bunu gerektirir; yoksa burada hata verir (sözleşme notu)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. Hat testi (ağırlıksız küçük modellerle, ~10 dk) — bu SMK-001 DEĞİLDİR
Gerçek veri ve gerçek ağırlık olmadan bütün kod yollarını sınar: pencereleme, dolgu yolu, kancalar, smoke test kodu, gömme çıkarımı (iki dolgu politikası, kaldığı yerden devam), problar (yerleştirilmiş sinyal bulunuyor, saf gürültü ≈ 0.5), D-035 fold başına seçimin dış test tahminlerinden bağımsızlığı, DeLong / Holm / hızlı bootstrap, ayrı meta veri raporu.

**Bu testlerin geçmesi, modellerin doğru yüklendiğini ya da anlamlı çıktı verdiğini göstermez.** Rastgele küçük modeller kullanılır. Resmi ağırlıkların doğruluğunu yalnız Bölüm 3'teki SMK-001 sınar. Test başarısız olursa hücre durur; `Tüm kontroller geçti` görmeden devam etme.

In [ ]:
sh(f'cd {REPO_DIR} && python tests/test_phase2.py')

## 2. Ağırlıkları Drive'a indir (yalnız ilk sefer; sonra atlanır)
- **PANNs** (Zenodo 3987831, CC BY 4.0): md5 Zenodo'nun kendi kaydından okunur ve `models/zenodo_3987831_md5.json`'a yazılır; sonraki oturumlarda dosyalar ağ bağlantısı olmadan bu kayda göre doğrulanır.
- **WavLM** (Hugging Face): çözümlenen commit hash'i `REVISION.txt`'ye yazılır; model ve özellik çıkarıcı `save_pretrained` ile Drive'a kopyalanır. Sonraki oturumlar internetteki sürüme değil bu kopyaya bakar.
- **BEATs:** elle (yukarıda). Burada yalnız varlığı ve boyutu kontrol edilir.

In [ ]:
PANNS = ['Cnn10_mAP=0.380.pth', 'Cnn14_mAP=0.431.pth', 'Cnn14_16k_mAP=0.438.pth']
MD5_FILE = MODELS / 'zenodo_3987831_md5.json'    # Zenodo kaydındaki md5'ler (ilk indirmede yazılır)


def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


known = json.loads(MD5_FILE.read_text()) if MD5_FILE.exists() else {}
if any(k not in known or not (MODELS / k).exists() for k in PANNS):     # yalnız gerekirse Zenodo'ya sor
    import requests
    resp = requests.get('https://zenodo.org/api/records/3987831', timeout=60)
    resp.raise_for_status()
    files = {f['key']: f for f in resp.json()['files']}
    for k in PANNS:
        algo, val = files[k]['checksum'].split(':', 1)
        assert algo == 'md5', files[k]['checksum']
        assert known.get(k, val) == val, f'{k}: Zenodo kaydındaki md5 değişmiş'
        known[k] = val
    for k in PANNS:
        dst = MODELS / k
        if dst.exists():
            continue
        tmp = dst.with_name(k + '.part')
        print('indiriliyor:', k)
        with requests.get(files[k]['links']['self'], stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(tmp, 'wb') as f:
                for b in r.iter_content(1 << 22):
                    f.write(b)
        if md5(tmp) != known[k]:
            tmp.unlink()
            raise RuntimeError(f'{k}: indirilen dosyanın md5\'i Zenodo kaydıyla uyuşmuyor (silindi; hücreyi yeniden çalıştır)')
        tmp.rename(dst)
    MD5_FILE.write_text(json.dumps(known, indent=2))
for k in PANNS:
    dst = MODELS / k
    assert md5(dst) == known[k], f'{k}: md5 Zenodo kaydıyla uyuşmuyor. Dosyayı sil ({dst}) ve hücreyi yeniden çalıştır.'
    print(k, 'md5 ✓', round(dst.stat().st_size / 1e6), 'MB')

In [ ]:
import json
from huggingface_hub import HfApi, snapshot_download
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
import transformers
for repo, name in [('microsoft/wavlm-base-plus', 'wavlm-base-plus'), ('microsoft/wavlm-large', 'wavlm-large')]:
    dst = MODELS / name
    # REVISION.txt EN SON yazılır: yalnız indirme + kayıt + bilgi dosyası tamamsa "mevcut" sayılır
    if (dst / 'REVISION.txt').exists() and (dst / 'download_info.json').exists():
        print(name, 'mevcut, revizyon', (dst / 'REVISION.txt').read_text().strip()); continue
    rev = HfApi().model_info(repo).sha
    snap = snapshot_download(repo, revision=rev)
    model, li = WavLMModel.from_pretrained(snap, output_loading_info=True)
    fe = Wav2Vec2FeatureExtractor.from_pretrained(snap)
    # transformers v5'te bu alanlar küme (set) olabilir → JSON için sıralı listeye çevir
    li = {k: sorted(map(str, v)) if isinstance(v, (set, list, tuple)) else v for k, v in li.items()}
    assert not li['missing_keys'], li['missing_keys'][:5]
    assert not li.get('mismatched_keys'), li['mismatched_keys'][:5]
    model.save_pretrained(dst); fe.save_pretrained(dst)
    (dst / 'download_info.json').write_text(json.dumps({'repo': repo, 'revision': rev,
        'unexpected_keys_at_download': li['unexpected_keys'], 'transformers': transformers.__version__,
        'return_attention_mask': bool(getattr(fe, 'return_attention_mask', False)), 'do_normalize': bool(fe.do_normalize)}, indent=2))
    (dst / 'REVISION.txt').write_text(rev + '\n')
    print(name, 'kaydedildi, revizyon', rev, '| do_normalize', fe.do_normalize, '| return_attention_mask',
          getattr(fe, 'return_attention_mask', None), '| beklenmeyen anahtar:', len(li['unexpected_keys']), li['unexpected_keys'][:5])

In [ ]:
beats = MODELS / 'BEATs_iter3_plus_AS2M.pt'
assert beats.exists(), ('BEATs ağırlığı yok. README tablosundaki "BEATs_iter3+ (AS2M)" (fine-tuned olmayan) bağlantısından indirip '
                        f'{beats} adıyla Drive\'a kaydet, sonra bu hücreyi yeniden çalıştır.')
size_mb = beats.stat().st_size / 1e6
assert size_mb > 100, (f'BEATs dosyası yalnız {size_mb:.1f} MB: büyük olasılıkla checkpoint değil, OneDrive\'ın HTML sayfası. '
                       'Tarayıcıdan "İndir" ile yeniden indir.')
print('BEATs mevcut,', round(size_mb), 'MB')

## 2b. Önbelleği yerel diske kopyala (sha256 doğrulamalı)

In [ ]:
# Drive → yerel disk (okuma hızı). Yarım kalmış kopya .part olarak kalır ve bir sonraki çalıştırmada yeniden kopyalanır.
info = json.loads((DRIVE_CACHE / 'cache_info.json').read_text())
LOCAL_CACHE.mkdir(exist_ok=True)
for name in ['audio_32k.f32', 'audio_16k.f32', 'index.csv']:
    dst = LOCAL_CACHE / name
    if dst.exists() and sha256(dst) != info['files_sha256'][name]:
        print(name, ': yerel kopya bozuk, yeniden kopyalanıyor'); dst.unlink()
    if not dst.exists():
        tmp = dst.with_name(name + '.part')
        shutil.copy(DRIVE_CACHE / name, tmp)
        tmp.rename(dst)
    assert sha256(dst) == info['files_sha256'][name], f'{name}: sha256 cache_info.json ile uyuşmuyor (Drive kopyası bozuk?)'
print('Önbellek yerel diskte ve doğrulandı.')

## 3. SMK-001 — gerçek checkpoint'lerle smoke test (~10 dk)
Her backbone için S1–S14, **resmi ağırlıklarla, GPU'da**. Rapor `reports/phase2/SMK-001_smoke.md`. (Bölüm 1'deki birim testleri `UNITTEST_random_init_smoke.*` adlı ayrı bir rapor üretir ve geçici klasörde silinir; repoya girmez.) Checkpoint sha256'ları `models/MANIFEST.json`'a yazılır; sonraki oturumlarda değişirse S1 FAIL verir. Rapor, doğrulanan kodun özetini (`code_sha256`) de içerir; 21 numaralı notebook bunu kontrol eder.

FAIL olsa da bu hücre durmaz: rapor bir sonraki hücrede GitHub'a gönderilir, karar en son hücrede verilir.

In [ ]:
import torch
DEVICE = 'cuda'
assert torch.cuda.is_available(), 'GPU yok: Runtime → Change runtime type → T4 GPU (SMK-001 S12/S13 GPU ister)'
rc = sh(f'cd {REPO_DIR} && python scripts/smoke_test_backbones.py --cache-dir {LOCAL_CACHE} --model-dir {MODELS} '
        f'--report-dir {REPO_DIR}/reports/phase2 --device {DEVICE}', check=False)
print('\nsmoke_test_backbones çıkış kodu:', rc, '(0 = tümü PASS; 1 = en az bir FAIL — karar son hücrede)')

## 4. Raporu GitHub'a gönder (yeni dal)
Yalnız `reports/phase2/` commit edilir (agrega; katılımcı ID'si yok, gönderimden önce kontrol edilir). Dal adı zaman damgalıdır (`phase2-smoke-results-YYYYMMDD-HHMMSS`); gönderimden sonra repo yeniden `main`'e döner. Notebook'u kaydetmeden önce çıktıları temizle (*Edit → Clear all outputs*).

In [ ]:
# Rapor commit'i her seferinde YENİ, zaman damgalı bir dala gider (aynı adlı dala ikinci gönderim reddedilirdi).
# Gönderim başarısız olursa commit yerel dalda kalır; bu hücreyi yeniden çalıştırmak aynı commit'i yeniden gönderir.
import re, time
import pandas as pd
BASE_BRANCH, PATHS, MSG = 'phase2-smoke-results', ['reports/phase2'], 'SMK-001: Faz 2 smoke test raporu (Colab)'
git('config', 'user.name', GITHUB_USER)
git('config', 'user.email', f'{GITHUB_USER}@users.noreply.github.com')
cur = git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip()
if cur.startswith(BASE_BRANCH + '-') and git('log', 'HEAD', '--not', '--remotes', '--oneline').stdout.strip():
    BRANCH = cur
    print('Önceki denemenin commit\'i yeniden gönderiliyor:', BRANCH)
else:
    assert cur == 'main', f"Beklenmeyen dal '{cur}': 0. bölümdeki repo hücresini yeniden çalıştır."
    git('add', '--', *PATHS)
    staged = git('diff', '--cached', '--name-only').stdout.split()
    assert staged, 'Gönderilecek değişiklik yok (rapor üretilmedi mi? yukarıdaki hücrenin çıktısına bak).'
    bad = [s for s in staged if not any(s == p or s.startswith(p.rstrip('/') + '/') for p in PATHS)]
    assert not bad, f'Beklenmeyen dosya: {bad}'
    ids = set(pd.read_csv(DERIVED / 'participant_context.csv').participant_id.astype(int))   # D-014 son kontrol
    for s in staged:
        txt = (REPO_DIR / s).read_text(errors='ignore')
        found = {int(m) for m in re.findall(r'(?<![\d.])(\d{6})(?![\d.])', txt)} & ids
        assert token not in txt and not found, f'{s}: token ya da katılımcı ID\'si içeriyor → commit edilmedi (D-014)'
    BRANCH = f"{BASE_BRANCH}-{time.strftime('%Y%m%d-%H%M%S', time.gmtime())}"
    git('checkout', '-b', BRANCH)
    git('commit', '-m', MSG)
    print('commit:', git('rev-parse', '--short', 'HEAD').stdout.strip(), '| dosyalar:', staged)
r = git('push', '-u', 'origin', BRANCH, check=False, show=True)
if r.returncode != 0:
    raise RuntimeError(f"Push başarısız. Commit yerel '{BRANCH}' dalında duruyor (kaybolmadı). Token/ağ sorununu giderip "
                       "BU hücreyi yeniden çalıştır; aynı commit yeniden gönderilir.")
git('checkout', 'main')
print(f'Gönderildi: {BRANCH}\nPR aç: https://github.com/{GITHUB_USER}/{REPO_NAME}/pull/new/{BRANCH}')

## 5. SMK-001 kararı

In [ ]:
rep = json.loads(git('show', f'{BRANCH}:reports/phase2/SMK-001_smoke.json').stdout)
assert rep['exp'] == 'SMK-001' and not rep['random_init']
status = {r['backbone']: r['status'] for r in rep['results']}
for r in rep['results']:
    bad = {k: v for k, v in r.get('checks', {}).items() if v['status'] == 'FAIL'}
    print(f"{r['backbone']:16s} {r['status']}", ('| ' + r['error']) if 'error' in r else '')
    for k, v in bad.items():
        print('    ', k, {kk: vv for kk, vv in v.items() if kk != 'status'})
missing = [b for b in BACKBONES if b not in status]
failed = [b for b, s in status.items() if s != 'PASS']
if missing or failed:
    raise RuntimeError(f'SMK-001 FAIL: {failed} eksik: {missing}. Rapor {BRANCH} dalına gönderildi (olumsuz sonuç korunur). '
                       '21 numaralı notebook\'a GEÇME; yukarıdaki değerleri paylaş, nedeni birlikte inceleyelim.')
print(f'\nSMK-001: 6 backbone PASS. Sıradaki adım: {BRANCH} PR\'ını aç ve main\'e birleştir; sonra 21 numaralı notebook.')